# Cirrhosis Outcomes Prediction — Teacher Notebook
**AOAI Task 4** · Multi-class classification · Tabular

### Contains the reference solution. Do not share with students.

| | |
|---|---|
| **Target** | `Status` ∈ {C, CL, D} |
| **Metric** | Multi-class log loss (lower is better) |

### The key insights
1. **Class weighting hurts log loss.** `class_weight='balanced'` makes the model over-predict the rare
`CL` class. Accuracy barely moves, but probabilities become miscalibrated, and log loss punishes that.
Removing it is the biggest single improvement.
2. **Missingness is structured, not random.** About 44% of rows are missing the *same block* of
columns (Drug, Ascites, Copper, ...), as in the original Mayo PBC study where non-trial patients
have fewer measurements. Trees handle `NaN` natively, so we keep it and add a "missing block" flag
instead of imputing medians (which invents values for almost half the rows).
3. **Shallower, slower trees + a second model** give a small extra gain.

### Notebook structure
1. Setup & data
2. The missing-value pattern
3. Features
4. Experiment: class weighting vs. log loss
5. Final models: LightGBM + XGBoost
6. Blend & submission
7. Grading guide

## 1. Setup & Data

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, accuracy_score

warnings.filterwarnings('ignore')

SEED    = 42
TARGET  = 'Status'
CLASSES = ['C', 'CL', 'D']
np.random.seed(SEED)

COMPETITION = 'cirrhosis-outcomes-prediction-task'
CANDIDATES  = [Path(f'/kaggle/input/competitions/{COMPETITION}'), Path(f'/kaggle/input/{COMPETITION}'), Path('data')]
DATA_DIR    = next(p for p in CANDIDATES if p.exists())

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
y     = train[TARGET].map({c: i for i, c in enumerate(CLASSES)}).values

prior = np.bincount(y) / len(y)
print(f'Train: {train.shape} | Test: {test.shape}')
print('Class prior:', dict(zip(CLASSES, prior.round(3))))
print(f'Log loss of predicting the prior for everyone: {log_loss(y, np.tile(prior, (len(y), 1))):.4f}')

## 2. The Missing-Value Pattern

In [ ]:
BLOCK = ['Drug', 'Ascites', 'Hepatomegaly', 'Spiders', 'Copper', 'Alk_Phos', 'SGOT']

missing = train.isnull().mean().sort_values(ascending=False)
print(missing[missing > 0].round(3))

block_missing = train[BLOCK].isnull().all(axis=1)
print(f'\nRows missing the whole block {BLOCK}: {block_missing.mean():.1%}')
print('\nOutcome distribution by block missingness:')
print(pd.crosstab(block_missing, train[TARGET], normalize='index').round(3))

**Talking point:** the block is missing for ~44% of patients, but the outcome mix is almost the same in
both groups, so missingness is *not* a strong signal on its own. What matters is not inventing values:
median imputation fills half the rows with the same fake number and blurs real relationships, while
trees can route `NaN` rows down their own branch.

## 3. Features

In [ ]:
def build_features(df):
    df = df.copy()
    binary_map = {'Y': 1, 'N': 0, 'M': 1, 'F': 0}
    for col in ['Sex', 'Ascites', 'Hepatomegaly', 'Spiders']:
        df[col] = df[col].map(binary_map)
    df['Drug']  = df['Drug'].map({'D-penicillamine': 1, 'Placebo': 0})
    df['Edema'] = df['Edema'].map({'N': 0, 'S': 1, 'Y': 2})

    df['block_missing'] = df[BLOCK].isnull().all(axis=1).astype(int)
    df['n_missing']     = df.isnull().sum(axis=1)
    df['Bili_Protime']  = df['Bilirubin'] * df['Prothrombin']
    df['ALBI_proxy']    = np.log1p(df['Bilirubin']) * 0.66 - df['Albumin'] * 0.085
    return df.drop(columns=['id', TARGET], errors='ignore')   # NaNs are kept on purpose

X      = build_features(train)
X_test = build_features(test)
print(f'Features: {X.shape[1]}')

## 4. Experiment: Class Weighting vs. Log Loss

Same features, same folds — only `class_weight` changes.

In [ ]:
N_FOLDS = 5
skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))

LGB_PARAMS = dict(
    objective='multiclass', n_estimators=5000, learning_rate=0.01,
    num_leaves=15, max_depth=5, min_child_samples=30,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbose=-1,
)

def cv_lgb(params):
    oof, test_pred = np.zeros((len(X), 3)), np.zeros((len(X_test), 3))
    for tr_idx, val_idx in folds:
        model = lgb.LGBMClassifier(**params)
        model.fit(X.iloc[tr_idx], y[tr_idx],
                  eval_set=[(X.iloc[val_idx], y[val_idx])],
                  callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[val_idx] = model.predict_proba(X.iloc[val_idx])
        test_pred   += model.predict_proba(X_test) / N_FOLDS
    return oof, test_pred

oof_bal, _ = cv_lgb({**LGB_PARAMS, 'class_weight': 'balanced'})
oof_lgb, test_lgb = cv_lgb(LGB_PARAMS)

for name, oof in [('balanced', oof_bal), ('no weighting', oof_lgb)]:
    print(f'{name:13s} | log loss {log_loss(y, oof):.4f} | accuracy {accuracy_score(y, oof.argmax(1)):.4f} '
          f'| mean P(CL) {oof[:, 1].mean():.3f} (true share {prior[1]:.3f})')

With balancing, the average predicted probability of `CL` is far above its real frequency
(≈0.044 vs. 0.027), and the other classes are distorted to compensate. Log loss evaluates
probabilities, not decisions, so the balanced model pays for it on every row.

## 5. Final Models: LightGBM + XGBoost

In [ ]:
XGB_PARAMS = dict(
    objective='multi:softprob', eval_metric='mlogloss', n_estimators=5000, learning_rate=0.01,
    max_depth=4, min_child_weight=3, subsample=0.8, colsample_bytree=0.5,
    early_stopping_rounds=200, random_state=SEED, n_jobs=-1,
)

oof_xgb, test_xgb = np.zeros((len(X), 3)), np.zeros((len(X_test), 3))
for tr_idx, val_idx in folds:
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X.iloc[tr_idx], y[tr_idx], eval_set=[(X.iloc[val_idx], y[val_idx])], verbose=False)
    oof_xgb[val_idx] = model.predict_proba(X.iloc[val_idx])
    test_xgb        += model.predict_proba(X_test) / N_FOLDS

print(f'LightGBM OOF log loss: {log_loss(y, oof_lgb):.4f}')
print(f'XGBoost  OOF log loss: {log_loss(y, oof_xgb):.4f}')

## 6. Blend & Submission

In [ ]:
weights = np.linspace(0, 1, 11)
scores  = [log_loss(y, w * oof_lgb + (1 - w) * oof_xgb) for w in weights]
best_w  = weights[int(np.argmin(scores))]
print(f'Best LightGBM weight: {best_w:.1f} → OOF log loss {min(scores):.4f}')

test_probs = best_w * test_lgb + (1 - best_w) * test_xgb
test_probs = test_probs / test_probs.sum(axis=1, keepdims=True)

submission = pd.DataFrame({'id': test['id']})
for i, cls in enumerate(CLASSES):
    submission[f'Status_{cls}'] = test_probs[:, i]
submission.to_csv('submission.csv', index=False)
print(f'Saved {len(submission)} predictions to submission.csv')
submission.head()

## 7. Grading Guide

### What separates strong submissions
1. Understanding that log loss scores **probabilities**: no class re-weighting, no clipping to 0/1
2. Investigating *why* values are missing instead of blindly imputing
3. Comparing every change under the same CV folds
4. Submitting probabilities in the right column order (`Status_C, Status_CL, Status_D`)